In [1]:
import numpy as np
import pandas as pd
import altair as alt


def simulate_data(n, std, mean):
    rng = np.random.default_rng()
    period = np.where(np.arange(n) < n // 2, "Calibration", "Monitoring")
    flagged = rng.choice([True, False], size=n, p=[0.15, 0.85])
    return pd.DataFrame({
        'value': rng.normal(loc=mean, scale=std, size=n),
        'index': np.arange(n) + 1,
        'period': period,
        'flagged': flagged,
    })

data = simulate_data(100, 1, 0)

In [2]:

from dataclasses import dataclass

@dataclass
class ShewhartLimits:
    center: float
    ucl: float
    lcl: float
    uwl: float
    lwl: float

limits = ShewhartLimits(center=0, ucl=6, lcl=-6, uwl=4, lwl=-4)

In [3]:
PRIMARY_COLOR = "steelblue"
ALERT_COLOR = "red"

# Base chart
base = alt.Chart(data).encode(
    x=alt.X('index:Q').title('Index'),
    y=alt.Y('value:Q', scale=alt.Scale(padding=20)).title('Value'),
)

# Data chart
data["status"] = np.select(
    condlist=[data["flagged"], data["period"] == "Calibration"],
    choicelist=["Violation", "Calibration"],
    default="Monitoring",
)

line = base.mark_line(color=PRIMARY_COLOR)

point = base.mark_point(size=100, filled=True, opacity=1).encode(
    fill=alt.Fill(
        'status',
        scale=alt.Scale(
            domain=["Calibration", "Monitoring", "Violation"],
            range=["white", PRIMARY_COLOR, ALERT_COLOR],
        ),
        legend=alt.Legend(title=None, orient='top'),
    ),
    stroke=alt.Stroke(
        'status',
        scale=alt.Scale(
            domain=["Calibration", "Monitoring", "Violation"],
            range=[PRIMARY_COLOR, PRIMARY_COLOR, ALERT_COLOR],
        ),
        legend=None,
    ),
)

# Limits
limits_df = pd.DataFrame({
    "value": [limits.ucl, limits.lcl, limits.uwl, limits.lwl, limits.center],
    "kind": ["Control limit", "Control limit", "Warning limit", "Warning limit", "Center line"],
})

limit_scale = alt.Scale(
    domain=["Control limit", "Warning limit", "Center line"],
    range=[ALERT_COLOR, "orange", "black"],
)

limit_lines = alt.Chart(limits_df).mark_rule().encode(
    y="value:Q",
    color=alt.Color("kind:N", scale=limit_scale, legend=alt.Legend(title=None, orient='top')),
)

# Construct
(limit_lines + line + point).configure_axis(grid=False, titleFontSize=20, labelFontSize=20).properties(width=1000, height=400)

alt.LayerChart(...)

In [4]:
def apply_shift(values, onset, size):
    values = values.copy()
    values[onset:] += size
    return values

def apply_drift(values, onset, multiplier):
    values = values.copy()
    n = len(values) - onset
    offset = np.arange(1, n + 1) * multiplier
    values[onset:] += offset
    return values

def apply_variance_increase(values, onset, multiplier, center):
    values = values.copy()
    values[onset:] = center + (values[onset:] - center) * multiplier
    return values


def simulate_data(n, std, mean):
    rng = np.random.default_rng()
    return pd.DataFrame({
        'value': rng.normal(loc=mean, scale=std, size=n),
        'index': np.arange(n) + 1
    })


In [5]:
np.arange(1, 10) * 2

array([ 2,  4,  6,  8, 10, 12, 14, 16, 18])

In [6]:
rng = np.random.default_rng()
values = rng.normal(size=10, scale=.5)

abs(apply_variance_increase(values, 5, 10, 0))

array([0.05650053, 0.43182292, 0.61243893, 0.05021935, 0.1883356 ,
       6.76727466, 1.06226037, 0.21462443, 1.88398779, 1.39118714])